In [1]:
import pandas as pd
from scipy import stats , optimize
import numpy as np
from tqdm import tqdm

In [2]:
data = pd.read_csv("Simulation-grp_project2 - Data.csv")

In [3]:
data = data.iloc[:-2]

In [4]:
data = data[['Interarrival Times', 'Service Times for Initial Phase',
       'Service Times for Placing Keyboard and Mouse',
       'Service Times for Assembling the Case (Aluminum Plates)']]

In [5]:
# ---------- Exponential fit ----------
# usually fix loc=0 for times
def expo_test(data):
    loc, scale = stats.expon.fit(data, floc=0)
    ks_stat, ks_p = stats.kstest(data, 'expon', args=(loc, scale))
    
    print("\nExponential fit:")
    print("lambda =", 1/scale)
    print("scale =", scale)
    print("KS p-value =", ks_p)
    
    if(ks_p > 0.05):
        print("Exponential distribution is valid")
    else:
        print("Exponential is not valid")

def gamma_gof(data):
    """
    Fit a Gamma distribution with loc fixed at 0
    and run a KS goodness-of-fit test.

    Returns:
        shape, scale, log_likelihood, ks_stat, ks_pvalue
    """
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]

    if np.any(data <= 0):
        raise ValueError("Gamma fit with loc=0 requires all data > 0.")

    # Fit gamma: returns (shape, loc, scale)
    shape, loc, scale = stats.gamma.fit(data, floc=0)

    # Log-likelihood
    log_likelihood = np.sum(stats.gamma.logpdf(data, a=shape, loc=loc, scale=scale))

    # KS test against fitted gamma
    ks_res = stats.kstest(data, 'gamma', args=(shape, loc, scale))

    return {
        "Shape": shape,
        "Scale": scale,
        "Log_Likelihood": log_likelihood,
        "Ks_Statistic": ks_res.statistic,
        "P_value": ks_res.pvalue
    }

def uniform_gof(data):
    """
    Fit a Uniform distribution and run a KS goodness-of-fit test.

    Returns:
        lower bound, upper bound, log_likelihood, ks_stat, ks_pvalue
    """
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]

    # Fit uniform: returns (loc, scale)
    loc, scale = stats.uniform.fit(data)

    # Log-likelihood
    log_likelihood = np.sum(stats.uniform.logpdf(data, loc=loc, scale=scale))

    # KS test against fitted uniform
    ks_res = stats.kstest(data, 'uniform', args=(loc, scale))

    return {
        "Lower_Bound": loc,
        "Upper_Bound": loc + scale,
        "Scale": scale,
        "Log_Likelihood": log_likelihood,
        "Ks_Statistic": ks_res.statistic,
        "P_value": ks_res.pvalue
    }

def lognormal_gof(data):
    """
    Fit a Lognormal distribution with loc fixed at 0
    and run a KS goodness-of-fit test.

    Returns:
        shape, scale, log_likelihood, ks_stat, ks_pvalue
    """
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]

    if np.any(data <= 0):
        raise ValueError("Lognormal fit with loc=0 requires all data > 0.")

    # Fit lognormal: returns (shape, loc, scale)
    shape, loc, scale = stats.lognorm.fit(data, floc=0)

    # Log-likelihood
    log_likelihood = np.sum(
        stats.lognorm.logpdf(data, s=shape, loc=loc, scale=scale)
    )

    # KS test against fitted lognormal
    ks_res = stats.kstest(data, 'lognorm', args=(shape, loc, scale))

    return {
        "Shape": shape,
        "Scale": scale,
        "Log_Likelihood": log_likelihood,
        "Ks_Statistic": ks_res.statistic,
        "P_value": ks_res.pvalue
    }

def triangular_gof(data):
    """
    Fit a triangular distribution and run a KS goodness-of-fit test.

    Returns:
        lower bound, mode, upper bound, log_likelihood, ks_stat, ks_pvalue
    """
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]

    # Fit triang: returns (c, loc, scale)
    c, loc, scale = stats.triang.fit(data)

    mode = loc + c * scale
    upper = loc + scale

    # Log-likelihood
    log_likelihood = np.sum(stats.triang.logpdf(data, c, loc=loc, scale=scale))

    # KS test against fitted triangular
    ks_res = stats.kstest(data, 'triang', args=(c, loc, scale))

    return {
        "Lower_Bound": loc,
        "Mode": mode,
        "Upper_Bound": upper,
        "C": c,
        "Scale": scale,
        "Log_Likelihood": log_likelihood,
        "Ks_Statistic": ks_res.statistic,
        "P_value": ks_res.pvalue
    }

def truncated_normal_gof(data):
    """
    Fit a left-truncated normal distribution on [0, inf)
    and run a KS goodness-of-fit test.

    Returns:
        mu, sigma, log_likelihood, ks_stat, ks_pvalue
    """
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]

    if np.any(data < 0):
        raise ValueError("Left-truncated normal at 0 requires all data >= 0.")

    # Negative log-likelihood
    def nll(params):
        mu, log_sigma = params
        sigma = np.exp(log_sigma)   # ensures sigma > 0
        a = (0 - mu) / sigma
        return -np.sum(
            stats.truncnorm.logpdf(data, a=a, b=np.inf, loc=mu, scale=sigma)
        )

    # starting values
    mu0 = np.mean(data)
    sigma0 = np.std(data, ddof=1)
    sigma0 = max(sigma0, 1e-8)

    res = optimize.minimize(
        nll,
        x0=[mu0, np.log(sigma0)],
        method="L-BFGS-B"
    )

    mu_hat = res.x[0]
    sigma_hat = np.exp(res.x[1])
    a_hat = (0 - mu_hat) / sigma_hat

    # Log-likelihood
    log_likelihood = np.sum(
        stats.truncnorm.logpdf(data, a=a_hat, b=np.inf, loc=mu_hat, scale=sigma_hat)
    )

    # KS test
    cdf_hat = lambda x: stats.truncnorm.cdf(
        x, a=a_hat, b=np.inf, loc=mu_hat, scale=sigma_hat
    )
    ks_res = stats.kstest(data, cdf_hat)

    return {
        "Mu": mu_hat,
        "Sigma": sigma_hat,
        "Log_Likelihood": log_likelihood,
        "Ks_Statistic": ks_res.statistic,
        "P_value": ks_res.pvalue
    }

def fit_gamma_mixture(data, n_components=2, max_iter=300, tol=1e-6, random_state=42):
    """
    Fit a K-component Gamma mixture using EM.
    Each component is Gamma(shape_k, scale_k) with loc=0.
    """
    rng = np.random.default_rng(random_state)

    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]

    if np.any(data <= 0):
        raise ValueError("Gamma mixture requires all data > 0.")

    n = len(data)
    K = n_components

    if K < 1:
        raise ValueError("n_components must be at least 1.")
    if n < K:
        raise ValueError("Number of data points must be at least n_components.")

    # ----- initialization -----
    sorted_data = np.sort(data)
    chunks = np.array_split(sorted_data, K)

    shapes = np.zeros(K)
    scales = np.zeros(K)
    weights = np.ones(K) / K

    for k in range(K):
        chunk = chunks[k]
        if len(chunk) < 2:
            chunk = data[rng.choice(n, size=min(max(2, n // K), n), replace=False)]

        try:
            a, _, s = stats.gamma.fit(chunk, floc=0)
        except Exception:
            m = np.mean(chunk)
            v = np.var(chunk, ddof=1) if len(chunk) > 1 else max(m**2, 1e-6)
            a = max(m**2 / max(v, 1e-8), 1e-3)
            s = max(v / max(m, 1e-8), 1e-3)

        shapes[k] = max(a, 1e-6)
        scales[k] = max(s, 1e-6)

    loglik_history = []
    responsibilities = np.zeros((n, K))

    def weighted_gamma_mle(x, w, a_init, s_init):
        w = np.asarray(w, dtype=float)

        if np.sum(w) <= 0:
            return a_init, s_init

        def neg_weighted_loglik(params):
            log_a, log_s = params
            a = np.exp(log_a)
            s = np.exp(log_s)
            ll = np.sum(w * stats.gamma.logpdf(x, a=a, loc=0, scale=s))
            return -ll

        res = optimize.minimize(
            neg_weighted_loglik,
            x0=[np.log(max(a_init, 1e-8)), np.log(max(s_init, 1e-8))],
            method="L-BFGS-B"
        )

        a_hat = np.exp(res.x[0])
        s_hat = np.exp(res.x[1])
        return a_hat, s_hat

    for _ in range(max_iter):
        # E-step
        component_densities = np.zeros((n, K))
        for k in range(K):
            component_densities[:, k] = (
                weights[k] * stats.gamma.pdf(data, a=shapes[k], loc=0, scale=scales[k])
            )

        row_sums = component_densities.sum(axis=1, keepdims=True)
        row_sums = np.maximum(row_sums, 1e-300)
        responsibilities = component_densities / row_sums

        # log-likelihood
        loglik = np.sum(np.log(row_sums[:, 0]))
        loglik_history.append(loglik)

        # convergence check
        if len(loglik_history) > 1:
            if abs(loglik_history[-1] - loglik_history[-2]) < tol:
                break

        # M-step
        weights = responsibilities.mean(axis=0)
        weights = weights / weights.sum()

        for k in range(K):
            shapes[k], scales[k] = weighted_gamma_mle(
                data,
                responsibilities[:, k],
                shapes[k],
                scales[k]
            )

    n_params = 3 * K - 1
    aic = 2 * n_params - 2 * loglik_history[-1]
    bic = n_params * np.log(n) - 2 * loglik_history[-1]

    return {
        "Weights": weights,
        "Shapes": shapes,
        "Scales": scales,
        "Log_Likelihood": loglik_history[-1],
        "LogLik_History": loglik_history,
        "Responsibilities": responsibilities,
        "AIC": aic,
        "BIC": bic,
        "N_Parameters": n_params,
        "N_Components": K
    }


def gamma_mixture_cdf(x, result):
    x = np.asarray(x, dtype=float)
    weights = result["Weights"]
    shapes = result["Shapes"]
    scales = result["Scales"]

    cdf = np.zeros_like(x, dtype=float)
    for k in range(len(weights)):
        cdf += weights[k] * stats.gamma.cdf(x, a=shapes[k], loc=0, scale=scales[k])
    return cdf


def sample_gamma_mixture(n, result, random_state=None):
    rng = np.random.default_rng(random_state)

    weights = result["Weights"]
    shapes = result["Shapes"]
    scales = result["Scales"]
    K = len(weights)

    components = rng.choice(K, size=n, p=weights)
    samples = np.empty(n)

    for k in range(K):
        idx = (components == k)
        nk = np.sum(idx)
        if nk > 0:
            samples[idx] = stats.gamma.rvs(
                a=shapes[k], loc=0, scale=scales[k], size=nk, random_state=rng
            )

    return samples


def gamma_mixture_gof(data, n_components=2, n_boot=200, max_iter=300, tol=1e-6, random_state=42):
    """
    Fit a gamma mixture, compute KS statistic, and estimate p-value by bootstrap.
    """
    rng = np.random.default_rng(random_state)

    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]

    if np.any(data <= 0):
        raise ValueError("Gamma mixture requires all data > 0.")

    n = len(data)

    # fit original data
    fit_res = fit_gamma_mixture(
        data,
        n_components=n_components,
        max_iter=max_iter,
        tol=tol,
        random_state=random_state
    )

    cdf_hat = lambda x: gamma_mixture_cdf(x, fit_res)
    ks_obs = stats.kstest(data, cdf_hat).statistic

    # bootstrap p-value
    ks_boot = np.empty(n_boot)

    for b in tqdm(range(n_boot)):
        sim_data = sample_gamma_mixture(
            n,
            fit_res,
            random_state=rng.integers(1, 10**9)
        )

        sim_fit = fit_gamma_mixture(
            sim_data,
            n_components=n_components,
            max_iter=max_iter,
            tol=tol,
            random_state=rng.integers(1, 10**9)
        )

        sim_cdf = lambda x: gamma_mixture_cdf(x, sim_fit)
        ks_boot[b] = stats.kstest(sim_data, sim_cdf).statistic

    p_value = np.mean(ks_boot >= ks_obs)

    fit_res["Ks_Statistic"] = ks_obs
    fit_res["P_value"] = p_value
    fit_res["Bootstrap_KS"] = ks_boot

    return fit_res

In [29]:
expo_test(data["Interarrival Times"])


Exponential fit:
lambda = 0.15421185963185252
scale = 6.484585571999999
KS p-value = 0.259572262138975
Exponential distribution is valid


In [30]:
expo_test(data["Service Times for Initial Phase"])


Exponential fit:
lambda = 0.23541635115914702
scale = 4.24779330355
KS p-value = 0.46187709192093596
Exponential distribution is valid


In [31]:
uniform_gof(data["Service Times for Placing Keyboard and Mouse"])

{'Lower_Bound': 4.55498,
 'Upper_Bound': 14.577000000000002,
 'Scale': 10.022020000000001,
 'Log_Likelihood': np.float64(-460.95693442903996),
 'Ks_Statistic': np.float64(0.06880173857166516),
 'P_value': np.float64(0.2867807604648128)}

In [39]:
from scipy import stats
x = data["Service Times for Assembling the Case (Aluminum Plates)"].astype(float).to_numpy()

c, loc, scale = stats.weibull_min.fit(x, floc=0)   # loc fixed at 0
ks_stat, p_value = stats.kstest(x, "weibull_min", args=(c, loc, scale))

print(c, loc, scale)
print(ks_stat, p_value)

3.499696543834821 0 3.8454895128345843
0.061813814331123484 0.41271229139082766


In [6]:
# column name
col = "Service Times for Assembling the Case (Aluminum Plates)"

# use only the raw observations, excluding the last two summary rows
x = pd.to_numeric(data[col], errors="coerce").dropna().to_numpy()


n = len(x)
sample_mean = np.mean(x)
sample_std = np.std(x, ddof=1)   # sample standard deviation
cv = sample_std / sample_mean

# skewness
sample_skewness = stats.skew(x, bias=False)

# kurtosis
# fisher=True  -> excess kurtosis
# fisher=False -> ordinary kurtosis
excess_kurtosis = stats.kurtosis(x, fisher=True, bias=False)
ordinary_kurtosis = stats.kurtosis(x, fisher=False, bias=False)

# -----------------------------
# 2-parameter Weibull fit
# -----------------------------
# loc fixed at 0
shape, loc, scale = stats.weibull_min.fit(x, floc=0)

# KS test against fitted Weibull
ks_stat, ks_pvalue = stats.kstest(x, "weibull_min", args=(shape, loc, scale))

# -----------------------------
# Print results
# -----------------------------
print("Column:", col)
print("n =", n)
print()

print("Sample statistics")
print("Mean =", sample_mean)
print("Std Dev =", sample_std)
print("Coefficient of Variation =", cv)
print("Skewness =", sample_skewness)
print("Excess Kurtosis =", excess_kurtosis)
print("Ordinary Kurtosis =", ordinary_kurtosis)
print()

print("2-parameter Weibull fit")
print("Shape (alpha) =", shape)
print("Scale (beta) =", scale)
print("Minimum / loc =", loc)
print()

print("KS test")

Column: Service Times for Assembling the Case (Aluminum Plates)
n = 200

Sample statistics
Mean = 3.486047525
Std Dev = 1.07315295756799
Coefficient of Variation = 0.30784231995459954
Skewness = -0.17341262972441973
Excess Kurtosis = 0.33322314174127676
Ordinary Kurtosis = 3.3332231417412768

2-parameter Weibull fit
Shape (alpha) = 3.499696543834821
Scale (beta) = 3.8454895128345843
Minimum / loc = 0

KS test
